# Self-Correction Dataset Generation

For each MBPP task:

1. The student (`unsloth/Qwen3-0.6B`) generates 4 samples in one batched GPU call.
   Each output is canonicalized to `<think>...</think><execute>...</execute>`, with
   MBPP test cases appended when the student forgot them, and executed in the sandbox.
2. Failing samples are handed to the teacher (Gemini), which impersonates the
   student in first person and rewrites the turn. Teacher rescues run in parallel
   threads, up to 2 teacher attempts per failing sample.
3. Up to 2 examples are saved per task: at most one teacher-rescued and one
   student-first-try. When several samples share an outcome type, the one with
   the shortest reasoning trace is kept.
4. NoWait logits suppression (arXiv 2506.08343) bans reflective filler tokens
   ("Wait", "Hmm", "Alternatively", ...) so the student can't fall into reasoning
   loops; a thinking-budget cap is enforced in parallel.

Output is sharegpt-style JSONL (one conversation per line).

In [ ]:
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install --upgrade --force-reinstall typing_extensions>=4.13.0
!pip install --upgrade google-genai

In [ ]:
import os
import re
import ast
import sys
import io
import json
import time
import contextlib
import traceback
import resource
import threading
import ctypes
from getpass import getpass

import torch
from tqdm import tqdm
from unsloth import FastLanguageModel
from datasets import load_dataset, concatenate_datasets

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
class TimeoutException(Exception):
    pass


def _async_raise_in_thread(tid, exc_type):
    """
    Raise `exc_type` inside the thread identified by `tid`. Returns True on success.
    Used as a thread-safe alternative to signal.alarm() which only works in the
    main thread of the main interpreter.
    """
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(
        ctypes.c_ulong(tid), ctypes.py_object(exc_type)
    )
    if res == 0:
        return False
    if res > 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(ctypes.c_ulong(tid), None)
        return False
    return True


def analyze_assertion_failure(line_code, context):
    try:
        tree = ast.parse(line_code.strip())
        if not isinstance(tree.body[0], ast.Assert):
            return None
        assert_node = tree.body[0]
        node = assert_node.test
        if isinstance(node, ast.Compare):
            left_node = node.left
            right_node = node.comparators[0]
            left_val = eval(compile(ast.Expression(body=left_node), filename="<string>", mode="eval"), context)
            right_val = eval(compile(ast.Expression(body=right_node), filename="<string>", mode="eval"), context)
            return {
                "actual": left_val,
                "expected": right_val,
                "actual_repr": ast.unparse(left_node),
                "expected_repr": ast.unparse(right_node),
            }
    except Exception:
        return None
    return None


def execute_code_with_feedback(code_str, cases=None, timeout_seconds=5, memory_limit_mb=4096):
    if cases is None:
        cases = []
    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()
    full_execution_script = f"{code_str}\n\n" + "\n".join(cases)

    target_tid = threading.get_ident()
    timer = threading.Timer(
        timeout_seconds,
        lambda: _async_raise_in_thread(target_tid, TimeoutException),
    )
    timer.daemon = True
    timer.start()

    apply_mem_limit = threading.current_thread() is threading.main_thread()
    soft = hard = None
    if apply_mem_limit:
        soft, hard = resource.getrlimit(resource.RLIMIT_AS)
        try:
            resource.setrlimit(resource.RLIMIT_AS, (memory_limit_mb * 1024 * 1024, hard))
        except ValueError:
            apply_mem_limit = False

    local_scope = {}
    try:
        with contextlib.redirect_stdout(stdout_capture), contextlib.redirect_stderr(stderr_capture):
            exec(full_execution_script, local_scope)
        timer.cancel()
        return True, f"Execution Successful. Stdout:\n{stdout_capture.getvalue()}", "None"

    except TimeoutException:
        timer.cancel()
        return False, "Execution Failed: Time Limit Exceeded (Possible Infinite Loop)", "Runtime"
    except MemoryError:
        timer.cancel()
        return False, "Execution Failed: Memory Limit Exceeded", "Runtime"
    except Exception:
        timer.cancel()
        exc_type, exc_value, exc_traceback = sys.exc_info()
        exc_name = exc_type.__name__
        tb_frames = traceback.extract_tb(exc_traceback)
        error_line_num = "Unknown"
        offending_line_code = "Could not extract line."

        if hasattr(exc_value, "lineno"):
            error_line_num = exc_value.lineno
        elif tb_frames:
            for frame in reversed(tb_frames):
                if frame.filename == "<string>":
                    error_line_num = frame.lineno
                    break

        if isinstance(error_line_num, int):
            script_lines = full_execution_script.splitlines()
            if 0 <= error_line_num - 1 < len(script_lines):
                offending_line_code = script_lines[error_line_num - 1].strip()

        details = str(exc_value)
        rich_feedback = ""
        if exc_name == "AssertionError":
            analysis = analyze_assertion_failure(offending_line_code, local_scope)
            if analysis:
                act_str = str(analysis["actual"])
                exp_str = str(analysis["expected"])
                if len(act_str) > 500: act_str = act_str[:500] + "... (truncated)"
                if len(exp_str) > 500: exp_str = exp_str[:500] + "... (truncated)"
                rich_feedback = (
                    f"\nAnalysis:\n   Input/Call: {analysis['actual_repr']}\n"
                    f"   Expected: {exp_str}\n   Actual:    {act_str}"
                )
        error_msg = f"Error Type: {exc_name}\nLine {error_line_num}: {offending_line_code}\nDetails: {details}{rich_feedback}"
        return False, error_msg, "Logical" if exc_name == "AssertionError" else "Runtime"
    finally:
        timer.cancel()
        if apply_mem_limit and soft is not None:
            try:
                resource.setrlimit(resource.RLIMIT_AS, (soft, hard))
            except ValueError:
                pass

In [ ]:
from huggingface_hub import login

hf_token = getpass("Hugging Face token: ")
login(token=hf_token)

GEMINI_API_KEY = getpass("Gemini API key: ")
TEACHER_MODEL = "gemini-3.1-flash-lite"

In [ ]:
STUDENT_MODEL_ID = "unsloth/Qwen3-0.6B"

student_model, student_tokenizer = FastLanguageModel.from_pretrained(
    STUDENT_MODEL_ID,
    max_seq_length=5120,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(student_model)

if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

TEMPERATURE         = 0.6
TOP_P               = 0.95
TOP_K               = 20
MIN_P               = 0.0
REPETITION_PENALTY  = 1.05
MAX_NEW_TOKENS      = 6144
MAX_THINKING_TOKENS = 4096

THINK_START_ID = student_tokenizer.convert_tokens_to_ids("<think>")
THINK_END_ID   = student_tokenizer.convert_tokens_to_ids("</think>")
student_model.generation_config.begin_thinking_token_id = THINK_START_ID
student_model.generation_config.end_thinking_token_id   = THINK_END_ID

In [ ]:
STUDENT_SYSTEM_PROMPT = """You are a Self-Correcting Python Agent. Solve the problem; the sandbox tool will verify your code.

### OUTPUT FORMAT
After your reasoning, output ONE <execute> block containing the function plus the user's assert statements appended verbatim. Nothing else after it.

Example:
<execute>
def add(a, b):
    return a + b

assert add(2, 3) == 5
</execute>

### RULES
- Exactly one <execute> block per turn. Function and asserts together, never split into separate blocks.
- On a debugging turn: quote Expected vs Actual from the sandbox tool feedback in your reasoning and identify the root cause before writing the corrected <execute> block.
"""

TEACHER_SYSTEM_PROMPT = """You are continuing your own attempt as a Self-Correcting Python Agent. Your previous code failed in the sandbox -- you must now write the next turn IN FIRST PERSON, as if you (the same agent) are picking up where you left off.

### YOUR ROLE
- **Own the mistake.** Assume the error in the previous `<execute>` block is YOUR error. Do not refer to "the student" or "the previous code" as something separate -- it is your code.
- **Build on your prior `<think>`.** Re-read your earlier reasoning. Pinpoint exactly which assumption or step was wrong, and say so.
- **Be forensic.** Quote the Expected vs Actual values from the feedback. Diagnose the root cause in concrete terms ("I assumed X, but the feedback shows Y, so my logic for Z was wrong").
- **Then fix and retry.** Write a corrected `<think>` block followed by a new `<execute>` block. Append the original `assert` statements at the end of your `<execute>` code so the sandbox can verify.

### OUTPUT FORMAT (STRICT)
<think>
[Forensic analysis of the error in first person + fix plan]
</think>
<execute>
[Corrected Python code with the original assert statements appended]
</execute>

### CONSTRAINTS
- Stay in first person throughout `<think>`. Never break character.
- Do not output anything outside the `<think>` and `<execute>` tags.
- Do not fabricate `<tool_response>` tags -- the sandbox provides them.
"""

In [ ]:
from transformers import LogitsProcessor, LogitsProcessorList


class ThinkingBudgetLogitsProcessor(LogitsProcessor):
    """Hard-cap the length of <think>...</think>. Use a fresh instance per generate() call."""

    def __init__(self, think_start_id, think_end_id, budget):
        self.think_start_id = think_start_id
        self.think_end_id   = think_end_id
        self.budget         = budget
        self._open_at      = None
        self._initial_len  = None
        self._non_end_mask = None

    def __call__(self, input_ids, scores):
        batch_size, cur_len = input_ids.shape
        device = input_ids.device

        if self._open_at is None:
            self._open_at = torch.full((batch_size,), -1, dtype=torch.long, device=device)
            self._initial_len = cur_len
            non_end = torch.ones(scores.shape[-1], dtype=torch.bool, device=device)
            non_end[self.think_end_id] = False
            self._non_end_mask = non_end
        elif cur_len > self._initial_len:
            last_tok = input_ids[:, -1]
            opens  = (last_tok == self.think_start_id)
            closes = (last_tok == self.think_end_id) & (self._open_at >= 0)
            self._open_at = torch.where(opens,  torch.full_like(self._open_at, cur_len - 1), self._open_at)
            self._open_at = torch.where(closes, torch.full_like(self._open_at, -1),         self._open_at)

        is_open         = self._open_at >= 0
        tokens_in_think = (cur_len - 1) - self._open_at
        over_budget     = is_open & (tokens_in_think >= self.budget)

        mask = over_budget.unsqueeze(1) & self._non_end_mask.unsqueeze(0)
        scores = scores.masked_fill(mask, float("-inf"))
        scores[:, self.think_end_id] = torch.where(
            over_budget,
            torch.zeros_like(scores[:, self.think_end_id]),
            scores[:, self.think_end_id],
        )
        return scores


class SuppressReflectiveTokens(LogitsProcessor):
    """
    NoWait (arXiv 2506.08343): soft- or hard-suppress reflective tokens that
    trigger reasoning loops. Enumerates every single-token form of each word
    (with/without leading space, case variants) and biases their logits.
    Use bias=float('-inf') for a hard ban, or a finite negative (e.g. -8.0)
    for a soft nudge that still allows the word when it's strongly correct.
    """
    def __init__(self, tokenizer, words, bias=float("-inf")):
        bad_ids = set()
        for w in words:
            variants = {w, " " + w,
                        w.lower(), " " + w.lower(),
                        w.capitalize(), " " + w.capitalize()}
            for v in variants:
                toks = tokenizer.encode(v, add_special_tokens=False)
                if len(toks) == 1:
                    bad_ids.add(toks[0])
        self.bad_ids = sorted(bad_ids)
        self.bias = bias
        self._bias_vec = None

    def __call__(self, input_ids, scores):
        if not self.bad_ids:
            return scores
        if self._bias_vec is None:
            self._bias_vec = torch.zeros(scores.shape[-1], device=scores.device)
            self._bias_vec[torch.tensor(self.bad_ids, device=scores.device)] = self.bias
        return scores + self._bias_vec


NOWAIT_WORDS = ["Wait", "Hmm", "Alternatively", "Alternative", "However", "Oh", "Ah"]
NOWAIT_BIAS  = float("-inf")


def _student_logits_processors(max_thinking_tokens):
    """Fresh processor list per generate() call (state must not leak across calls)."""
    return LogitsProcessorList([
        ThinkingBudgetLogitsProcessor(THINK_START_ID, THINK_END_ID, max_thinking_tokens),
        SuppressReflectiveTokens(student_tokenizer, NOWAIT_WORDS, bias=NOWAIT_BIAS),
    ])


def generate_student_turn(messages, max_new_tokens=None, temperature=None,
                          max_thinking_tokens=None):
    """One generation call to the local Qwen student. Returns raw assistant text."""
    text = student_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = student_tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = student_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens if max_new_tokens is not None else MAX_NEW_TOKENS,
            temperature=temperature if temperature is not None else TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            min_p=MIN_P,
            repetition_penalty=REPETITION_PENALTY,
            do_sample=True,
            pad_token_id=student_tokenizer.pad_token_id,
            logits_processor=_student_logits_processors(
                max_thinking_tokens if max_thinking_tokens is not None else MAX_THINKING_TOKENS
            ),
        )
    new_tokens = out[:, inputs.input_ids.shape[1]:]
    return student_tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()


def generate_student_batch(messages, n_samples=4, max_new_tokens=None, temperature=None,
                           max_thinking_tokens=None):
    """Generate `n_samples` completions for the same prompt in one batched GPU call."""
    text = student_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = student_tokenizer(text, return_tensors="pt").to("cuda")
    input_len = inputs.input_ids.shape[1]
    with torch.no_grad():
        out = student_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens if max_new_tokens is not None else MAX_NEW_TOKENS,
            temperature=temperature if temperature is not None else TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            min_p=MIN_P,
            repetition_penalty=REPETITION_PENALTY,
            do_sample=True,
            num_return_sequences=n_samples,
            pad_token_id=student_tokenizer.pad_token_id,
            logits_processor=_student_logits_processors(
                max_thinking_tokens if max_thinking_tokens is not None else MAX_THINKING_TOKENS
            ),
        )
    new_tokens = out[:, input_len:]
    return [
        student_tokenizer.decode(seq, skip_special_tokens=True).strip()
        for seq in new_tokens
    ]


from google import genai
from google.genai import types

_gemini_client = None
def _get_gemini_client():
    global _gemini_client
    if _gemini_client is None:
        _gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    return _gemini_client


def generate_teacher_turn(history_messages, max_retries=3):
    """Call Gemini with the full conversation so far and return the teacher continuation."""
    client = _get_gemini_client()

    contents = []
    for m in history_messages:
        if m["role"] == "system":
            continue
        role = "user" if m["role"] == "user" else "model"
        contents.append(types.Content(role=role, parts=[types.Part.from_text(text=m["content"])]))

    config = types.GenerateContentConfig(
        system_instruction=TEACHER_SYSTEM_PROMPT,
        temperature=0.3,
        max_output_tokens=16384,
        thinking_config=types.ThinkingConfig(thinking_level="low"),
    )

    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model=TEACHER_MODEL,
                contents=contents,
                config=config,
            )
            return resp.text.strip()
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Gemini teacher failed after {max_retries} retries: {last_err}")


_THINK_RE   = re.compile(r"<think>(.*?)</think>", re.DOTALL)
_EXECUTE_RE = re.compile(r"<execute>(.*?)</execute>", re.DOTALL)
_MD_PY_RE   = re.compile(r"```python\s*(.*?)```", re.DOTALL)
_MD_ANY_RE  = re.compile(r"```\s*\n(.*?)```", re.DOTALL)


def _split_code_and_asserts(blocks):
    """Split a list of code blocks into (code_parts, assertion_parts).
    Assertion blocks are those whose first non-blank line starts with 'assert'."""
    code_parts, assertion_parts = [], []
    for block in blocks:
        lines = [l for l in block.split("\n") if l.strip()]
        if not lines:
            continue
        if lines[0].lstrip().startswith("assert"):
            assertion_parts.append(block)
        else:
            code_parts.append(block)
    return code_parts, assertion_parts


def _has_top_level_assert(code):
    for line in code.split("\n"):
        if line.lstrip().startswith("assert "):
            return True
    return False


def extract_code(response_text):
    """
    Pull executable code from <execute> tags or python markdown fences. When the
    response has multiple blocks (e.g. one with the function definition and one
    with the assertions), they are concatenated into a single script. Returns
    None if no code is found.
    """
    execute_blocks = [m.strip() for m in _EXECUTE_RE.findall(response_text)]
    py_blocks      = [m.strip() for m in _MD_PY_RE.findall(response_text)]
    any_blocks     = [m.strip() for m in _MD_ANY_RE.findall(response_text)]

    if execute_blocks:
        blocks = execute_blocks
    elif py_blocks:
        blocks = py_blocks
    elif any_blocks:
        blocks = any_blocks
    else:
        return None

    code_parts, assertion_parts = _split_code_and_asserts(blocks)
    if not code_parts and not assertion_parts:
        return None

    code = "\n\n".join(code_parts) if code_parts else ""
    if assertion_parts:
        asserts = "\n".join(assertion_parts)
        return code + "\n\n" + asserts if code else asserts
    return code


def canonicalize_assistant_turn(raw_text, cases=None):
    """
    Rebuild an assistant turn to strictly contain:
        <think>{think}</think><execute>{code}</execute>

    If the extracted code has no assert statements and `cases` is provided,
    the MBPP assertions are appended so the saved <execute> block always carries
    the tests inside it.
    """
    code = extract_code(raw_text)
    if code is None:
        return None

    if cases and not _has_top_level_assert(code):
        code = code + "\n\n" + "\n".join(cases)

    think_content = ""
    for m in _THINK_RE.finditer(raw_text):
        candidate = m.group(1).strip()
        if candidate:
            think_content = candidate
            break

    return f"<think>\n{think_content}\n</think>\n<execute>\n{code}\n</execute>"

In [ ]:
def generate_student_batch_multi_task(msgs_list, n_samples_per_task=4,
                                      max_new_tokens=None, temperature=None,
                                      max_thinking_tokens=None):
    """Batched student generation across multiple tasks.

    Args:
        msgs_list: list of `messages` (one per task). Different prompts.
        n_samples_per_task: samples per task (uses num_return_sequences).

    Returns:
        List[List[str]] of shape [len(msgs_list)][n_samples_per_task] -- raw
        assistant text per task, in input order.
    """
    texts = [
        student_tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in msgs_list
    ]

    # Left-pad for decoder-only generation. Restore after to avoid leaking state.
    orig_padding_side = student_tokenizer.padding_side
    student_tokenizer.padding_side = "left"
    try:
        inputs = student_tokenizer(
            texts, return_tensors="pt", padding=True
        ).to("cuda")
    finally:
        student_tokenizer.padding_side = orig_padding_side

    input_len = inputs.input_ids.shape[1]
    n_tasks   = len(texts)

    with torch.no_grad():
        out = student_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens if max_new_tokens is not None else MAX_NEW_TOKENS,
            temperature=temperature if temperature is not None else TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            min_p=MIN_P,
            repetition_penalty=REPETITION_PENALTY,
            do_sample=True,
            num_return_sequences=n_samples_per_task,
            pad_token_id=student_tokenizer.pad_token_id,
            logits_processor=_student_logits_processors(
                max_thinking_tokens if max_thinking_tokens is not None else MAX_THINKING_TOKENS
            ),
        )

    # HF interleaves as [task0_s0, task0_s1, ..., task0_sK-1, task1_s0, ...].
    new_tokens = out[:, input_len:]
    decoded = [
        student_tokenizer.decode(seq, skip_special_tokens=True).strip()
        for seq in new_tokens
    ]
    return [
        decoded[i * n_samples_per_task : (i + 1) * n_samples_per_task]
        for i in range(n_tasks)
    ]

In [ ]:
import concurrent.futures

N_SAMPLES_PER_TASK    = 4
MAX_TEACHER_ATTEMPTS  = 2
TEACHER_API_RETRY_MAX = 2


def _reasoning_token_count(sample):
    """Sum tokens of all <think> content across the sample's saved gpt turns."""
    total = 0
    for turn in sample.get("saved_conv", []):
        if turn["from"] == "gpt":
            for m in _THINK_RE.finditer(turn["value"]):
                total += len(student_tokenizer.encode(m.group(1), add_special_tokens=False))
    return total


def _trajectory_code_signature(sample):
    """Normalized concatenation of every <execute> code block across the sample's
    gpt turns. Two teacher rescues with the SAME initial error AND the SAME fix(es)
    collapse to one signature; differing errors or differing repairs stay distinct."""
    blocks = []
    for turn in sample.get("saved_conv", []):
        if turn["from"] == "gpt":
            code = extract_code(turn["value"])
            if code is not None:
                norm = "\n".join(ln.rstrip() for ln in code.strip().splitlines())
                blocks.append(norm)
    return "\u241e".join(blocks)


def _distinct_teacher_solved(teacher_solved):
    """Keep every DISTINCT teacher-rescued trajectory for a task (dedup by code
    signature), choosing the shortest-reasoning representative of each."""
    by_sig = {}
    for s in teacher_solved:
        sig = _trajectory_code_signature(s)
        cur = by_sig.get(sig)
        if cur is None or _reasoning_token_count(s) < _reasoning_token_count(cur):
            by_sig[sig] = s
    return list(by_sig.values())


def _run_teacher_chain(initial_msgs, initial_saved_conv, task,
                       max_teacher_attempts=MAX_TEACHER_ATTEMPTS):
    """Drive a teacher rescue starting from a conversation that already contains
    one failed assistant turn + its tool_response."""
    msgs = list(initial_msgs)
    saved_conv = list(initial_saved_conv)
    teacher_turns = 0

    for _ in range(max_teacher_attempts):
        teacher_turns += 1
        try:
            raw = generate_teacher_turn(msgs)
        except Exception as e:
            return {
                "status": "dropped",
                "drop_reason": f"teacher_generation_error: {e}",
                "saved_conv": saved_conv,
                "teacher_turns": teacher_turns,
            }

        code = extract_code(raw)
        if code is None:
            return {
                "status": "dropped",
                "drop_reason": "teacher_no_code",
                "saved_conv": saved_conv,
                "teacher_turns": teacher_turns,
            }

        saved_assistant = canonicalize_assistant_turn(raw, cases=task["tests"])
        msgs.append({"role": "assistant", "content": saved_assistant})
        saved_conv.append({"from": "gpt", "value": saved_assistant})

        ok, msg, _ = execute_code_with_feedback(code, task["tests"])
        if ok:
            saved_conv.append({
                "from": "human",
                "value": f"<tool_response>\nExecution Success\n{msg}\n</tool_response>",
            })
            return {
                "status": "solved_teacher",
                "drop_reason": None,
                "saved_conv": saved_conv,
                "teacher_turns": teacher_turns,
            }

        fb = f"<tool_response>\nExecution Failed\n{msg}\n</tool_response>"
        saved_conv.append({"from": "human", "value": fb})
        msgs.append({"role": "user", "content": fb})

    return {
        "status": "dropped",
        "drop_reason": f"unsolved_after_{1 + max_teacher_attempts}_turns",
        "saved_conv": saved_conv,
        "teacher_turns": teacher_turns,
    }


def _judge_student_sample(raw, base_msgs, base_saved_conv, task):
    """Classify one student raw output and prep teacher input if it failed."""
    code = extract_code(raw)
    if code is None:
        return {
            "student_raw": raw,
            "student_code": None,
            "outcome": "student_no_code",
            "student_sandbox_msg": None,
        }

    saved_assistant = canonicalize_assistant_turn(raw, cases=task["tests"])
    saved_conv = list(base_saved_conv) + [{"from": "gpt", "value": saved_assistant}]
    msgs       = list(base_msgs)       + [{"role": "assistant", "content": saved_assistant}]

    ok, msg, _ = execute_code_with_feedback(code, task["tests"])
    if ok:
        saved_conv.append({
            "from": "human",
            "value": f"<tool_response>\nExecution Success\n{msg}\n</tool_response>",
        })
        return {
            "student_raw": raw,
            "student_canon": saved_assistant,
            "student_code": code,
            "outcome": "solved_student",
            "student_sandbox_msg": msg,
            "saved_conv": saved_conv,
        }

    fb = f"<tool_response>\nExecution Failed\n{msg}\n</tool_response>"
    saved_conv.append({"from": "human", "value": fb})
    msgs.append({"role": "user", "content": fb})
    return {
        "student_raw": raw,
        "student_canon": saved_assistant,
        "student_code": code,
        "outcome": "student_failed",
        "student_sandbox_msg": msg,
        "_teacher_msgs": msgs,
        "_teacher_saved_conv": saved_conv,
    }


def _build_saved_example(sample, task, n_samples, temperature,
                         n_teacher_solved, n_student_solved, rescued):
    """Convert a chosen sample into the dataset-ready JSONL record."""
    if rescued:
        teacher_turns = sample.get("teacher_result", {}).get("teacher_turns", 0)
        attempts_used = 1 + teacher_turns
    else:
        attempts_used = 1
    return {
        "task_id": task["task_id"],
        "conversations": sample["saved_conv"],
        "metadata": {
            "attempts_used": attempts_used,
            "rescued_by_teacher": rescued,
            "n_samples_total": n_samples,
            "n_teacher_solved": n_teacher_solved,
            "n_student_solved": n_student_solved,
            "sample_temperature": temperature,
            "student_model": STUDENT_MODEL_ID,
            "teacher_model": TEACHER_MODEL,
        },
    }


def process_task_multisample(task,
                             n_samples=N_SAMPLES_PER_TASK,
                             temperature=None,
                             api_retry_max=TEACHER_API_RETRY_MAX,
                             verbose=False):
    """Run the full pipeline on one MBPP task. The returned audit dict's
    `saved_examples` field is a list of 0-2 dataset-ready records: at most
    one teacher-rescued and one student-first-try, each picked as the
    shortest-reasoning sample of its outcome type."""
    if temperature is None:
        temperature = TEMPERATURE

    user_msg = f"Problem: {task['prompt']}\n\nTest Cases:\n" + "\n".join(task["tests"])
    base_msgs = [
        {"role": "system", "content": STUDENT_SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    base_saved_conv = [
        {"from": "human", "value": user_msg},
    ]

    try:
        raws = generate_student_batch(
            base_msgs, n_samples=n_samples, temperature=temperature,
        )
    except Exception as e:
        return {
            "task_id": task["task_id"], "problem": task["prompt"], "tests": task["tests"],
            "status": "dropped",
            "drop_reason": f"student_batch_error: {e}",
            "n_samples": n_samples,
            "sample_temperature": temperature,
            "n_teacher_solved": 0,
            "n_student_solved": 0,
            "n_saved": 0,
            "samples": [],
            "saved_examples": [],
        }

    samples = [_judge_student_sample(r, base_msgs, base_saved_conv, task) for r in raws]

    pending = [s for s in samples if s["outcome"] == "student_failed"]
    if pending:
        with concurrent.futures.ThreadPoolExecutor(max_workers=len(pending)) as pool:
            futs = {
                pool.submit(
                    _run_teacher_chain,
                    s.pop("_teacher_msgs"),
                    s.pop("_teacher_saved_conv"),
                    task,
                ): s for s in pending
            }
            for fut in concurrent.futures.as_completed(futs):
                s = futs[fut]
                res = fut.result()
                s["teacher_result"] = res
                s["outcome"] = res["status"]
                if res["status"] == "solved_teacher":
                    s["saved_conv"] = res["saved_conv"]

    def any_teacher_solved():
        return any(s["outcome"] == "solved_teacher" for s in samples)

    for round_idx in range(api_retry_max):
        if any_teacher_solved():
            break
        retry_targets = [
            s for s in samples
            if s["outcome"] == "dropped"
            and s.get("teacher_result", {}).get("drop_reason") == "teacher_no_code"
        ]
        if not retry_targets:
            break

        if verbose:
            print(f"  [retry round {round_idx+1}] re-running {len(retry_targets)} teacher_no_code chain(s)")

        with concurrent.futures.ThreadPoolExecutor(max_workers=len(retry_targets)) as pool:
            jobs = {}
            for s in retry_targets:
                student_canon = s.get("student_canon") or canonicalize_assistant_turn(
                    s["student_raw"], cases=task["tests"]
                )
                fb = f"<tool_response>\nExecution Failed\n{s['student_sandbox_msg']}\n</tool_response>"
                msgs = list(base_msgs) + [
                    {"role": "assistant", "content": student_canon},
                    {"role": "user", "content": fb},
                ]
                saved_conv = list(base_saved_conv) + [
                    {"from": "gpt", "value": student_canon},
                    {"from": "human", "value": fb},
                ]
                jobs[pool.submit(_run_teacher_chain, msgs, saved_conv, task)] = s
            for fut in concurrent.futures.as_completed(jobs):
                s = jobs[fut]
                res = fut.result()
                s["teacher_result"] = res
                s["outcome"] = res["status"]
                if res["status"] == "solved_teacher":
                    s["saved_conv"] = res["saved_conv"]

    teacher_solved = [s for s in samples if s["outcome"] == "solved_teacher"]
    student_solved = [s for s in samples if s["outcome"] == "solved_student"]

    saved_examples = []
    # Keep every distinct teacher-corrected trajectory.
    for ts in _distinct_teacher_solved(teacher_solved):
        saved_examples.append(_build_saved_example(
            ts, task, n_samples, temperature,
            len(teacher_solved), len(student_solved), rescued=True,
        ))
    if student_solved:
        best = min(student_solved, key=_reasoning_token_count)
        saved_examples.append(_build_saved_example(
            best, task, n_samples, temperature,
            len(teacher_solved), len(student_solved), rescued=False,
        ))

    status = "saved" if saved_examples else "dropped"
    drop_reason = None if saved_examples else "all_samples_dropped"

    return {
        "task_id": task["task_id"],
        "problem": task["prompt"],
        "tests": task["tests"],
        "status": status,
        "drop_reason": drop_reason,
        "n_samples": n_samples,
        "sample_temperature": temperature,
        "n_teacher_solved": len(teacher_solved),
        "n_student_solved": len(student_solved),
        "n_saved": len(saved_examples),
        "samples": [
            {
                "outcome": s["outcome"],
                "teacher_drop_reason": s.get("teacher_result", {}).get("drop_reason"),
                "teacher_turns": s.get("teacher_result", {}).get("teacher_turns", 0),
                "student_sandbox_head": (s.get("student_sandbox_msg") or "").splitlines()[:1],
            }
            for s in samples
        ],
        "saved_examples": saved_examples,
    }

In [ ]:
def process_tasks_batch(tasks,
                        n_samples=N_SAMPLES_PER_TASK,
                        temperature=None,
                        api_retry_max=TEACHER_API_RETRY_MAX,
                        verbose=False):
    """Process a batch of MBPP tasks together.

    Student samples for all tasks come from ONE batched generate() call
    (n_tasks * n_samples sequences in flight). Teacher rescues are pooled
    across all failing samples from all tasks for maximum parallelism.

    Returns: list of audit dicts in the same order as `tasks`.
    """
    if temperature is None:
        temperature = TEMPERATURE

    tds = []
    for task in tasks:
        user_msg = f"Problem: {task['prompt']}\n\nTest Cases:\n" + "\n".join(task["tests"])
        tds.append({
            "task": task,
            "base_msgs": [
                {"role": "system", "content": STUDENT_SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            "base_saved_conv": [{"from": "human", "value": user_msg}],
            "samples": [],
        })

    try:
        all_raws = generate_student_batch_multi_task(
            [td["base_msgs"] for td in tds],
            n_samples_per_task=n_samples,
            temperature=temperature,
        )
    except Exception as e:
        return [
            {
                "task_id": td["task"]["task_id"], "problem": td["task"]["prompt"],
                "tests": td["task"]["tests"], "status": "dropped",
                "drop_reason": f"student_batch_error: {e}",
                "n_samples": n_samples, "sample_temperature": temperature,
                "n_teacher_solved": 0, "n_student_solved": 0, "n_saved": 0,
                "samples": [], "saved_examples": [],
            }
            for td in tds
        ]

    for td, raws in zip(tds, all_raws):
        td["samples"] = [
            _judge_student_sample(r, td["base_msgs"], td["base_saved_conv"], td["task"])
            for r in raws
        ]

    pending = [(td, s) for td in tds for s in td["samples"] if s["outcome"] == "student_failed"]
    if pending:
        with concurrent.futures.ThreadPoolExecutor(max_workers=min(len(pending), 32)) as pool:
            futs = {}
            for td, s in pending:
                futs[pool.submit(
                    _run_teacher_chain,
                    s.pop("_teacher_msgs"),
                    s.pop("_teacher_saved_conv"),
                    td["task"],
                )] = (td, s)
            for fut in concurrent.futures.as_completed(futs):
                td, s = futs[fut]
                res = fut.result()
                s["teacher_result"] = res
                s["outcome"] = res["status"]
                if res["status"] == "solved_teacher":
                    s["saved_conv"] = res["saved_conv"]

    for round_idx in range(api_retry_max):
        retry_jobs = []
        for td in tds:
            # Skip retries for tasks that already have a teacher solve
            if any(s["outcome"] == "solved_teacher" for s in td["samples"]):
                continue
            for s in td["samples"]:
                if (s["outcome"] == "dropped"
                        and s.get("teacher_result", {}).get("drop_reason") == "teacher_no_code"):
                    student_canon = s.get("student_canon") or canonicalize_assistant_turn(
                        s["student_raw"], cases=td["task"]["tests"]
                    )
                    fb = f"<tool_response>\nExecution Failed\n{s['student_sandbox_msg']}\n</tool_response>"
                    msgs = list(td["base_msgs"]) + [
                        {"role": "assistant", "content": student_canon},
                        {"role": "user", "content": fb},
                    ]
                    saved_conv = list(td["base_saved_conv"]) + [
                        {"from": "gpt", "value": student_canon},
                        {"from": "human", "value": fb},
                    ]
                    retry_jobs.append((td, s, msgs, saved_conv))
        if not retry_jobs:
            break
        if verbose:
            print(f"  [retry round {round_idx+1}] re-running {len(retry_jobs)} teacher_no_code chain(s) across batch")
        with concurrent.futures.ThreadPoolExecutor(max_workers=min(len(retry_jobs), 32)) as pool:
            jobs = {
                pool.submit(_run_teacher_chain, msgs, saved_conv, td["task"]): (td, s)
                for td, s, msgs, saved_conv in retry_jobs
            }
            for fut in concurrent.futures.as_completed(jobs):
                td, s = jobs[fut]
                res = fut.result()
                s["teacher_result"] = res
                s["outcome"] = res["status"]
                if res["status"] == "solved_teacher":
                    s["saved_conv"] = res["saved_conv"]

    audits = []
    for td in tds:
        samples = td["samples"]
        task    = td["task"]
        teacher_solved = [s for s in samples if s["outcome"] == "solved_teacher"]
        student_solved = [s for s in samples if s["outcome"] == "solved_student"]

        saved_examples = []
        # Keep every distinct teacher-corrected trajectory.
        for ts in _distinct_teacher_solved(teacher_solved):
            saved_examples.append(_build_saved_example(
                ts, task, n_samples, temperature,
                len(teacher_solved), len(student_solved), rescued=True,
            ))
        if student_solved:
            best = min(student_solved, key=_reasoning_token_count)
            saved_examples.append(_build_saved_example(
                best, task, n_samples, temperature,
                len(teacher_solved), len(student_solved), rescued=False,
            ))

        status      = "saved" if saved_examples else "dropped"
        drop_reason = None if saved_examples else "all_samples_dropped"

        audits.append({
            "task_id": task["task_id"], "problem": task["prompt"], "tests": task["tests"],
            "status": status, "drop_reason": drop_reason,
            "n_samples": n_samples, "sample_temperature": temperature,
            "n_teacher_solved": len(teacher_solved),
            "n_student_solved": len(student_solved),
            "n_saved": len(saved_examples),
            "samples": [
                {
                    "outcome": s["outcome"],
                    "teacher_drop_reason": s.get("teacher_result", {}).get("drop_reason"),
                    "teacher_turns": s.get("teacher_result", {}).get("teacher_turns", 0),
                    "student_sandbox_head": (s.get("student_sandbox_msg") or "").splitlines()[:1],
                }
                for s in samples
            ],
            "saved_examples": saved_examples,
        })
    return audits

In [ ]:
DEMO_TASK = {
    "task_id": 990,
    "prompt": "Write a python function to find the difference between highest and least frequencies in a given array.",
    "tests": [
      "assert find_Diff([1,1,2,2,7,8,4,5,1,4],10) == 2",
      "assert find_Diff([1,7,9,2,3,3,1,3,3],9) == 3",
      "assert find_Diff([1,2,1,2],4) == 0"
    ],
}

audit = process_task_multisample(DEMO_TASK)

with open(f"audit_{DEMO_TASK['task_id']}.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

## Bulk generation over MBPP

Generates the dataset across the configured task range, writing dataset-ready
examples to `OUTPUT_PATH` and a per-task audit log to `AUDIT_PATH`. Each task
is processed independently and may contribute up to 2 saved examples. The cell
is resume-safe: task IDs already present in the audit file are skipped.

In [ ]:
OUTPUT_PATH = "/mnt/training/mbpp_0.6B_train_A.jsonl"
AUDIT_PATH  = "/mnt/training/mbpp_0.6B_train_Audit_A.jsonl"
INCLUDE_ALL_MBPP = False
START_TASK_ID    = 601
END_TASK_ID      = 789
N_SAMPLES_PER_TASK = 4
TASKS_PER_BATCH    = 8
SAMPLE_TEMPERATURE = 0.7
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

already_done = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                already_done.add(json.loads(line)["task_id"])
            except Exception:
                pass
print(f"Resuming: {len(already_done)} tasks already processed (from OUTPUT_PATH)")

mbpp = load_dataset("google-research-datasets/mbpp", "full")
full_mbpp = concatenate_datasets([mbpp[s] for s in ["train", "test", "validation", "prompt"]])

if INCLUDE_ALL_MBPP:
    target_rows = [r for r in full_mbpp if r["task_id"] not in already_done]
else:
    target_rows = [
        r for r in full_mbpp
        if START_TASK_ID <= r["task_id"] <= END_TASK_ID
        and r["task_id"] not in already_done
    ]
print(f"Target tasks this run: {len(target_rows)} "
      f"(tasks_per_batch={TASKS_PER_BATCH}, n_samples={N_SAMPLES_PER_TASK}, "
      f"temperature={SAMPLE_TEMPERATURE})")

stats = {
    "tasks_saved":             0,
    "tasks_dropped":           0,
    "examples_teacher":        0,
    "examples_student":        0,
    "total_teacher_solves":    0,
    "total_student_solves":    0,
    "total_samples_run":       0,
    "teacher_no_code_retries": 0,
}

with open(OUTPUT_PATH, "a", encoding="utf-8") as f_out, \
     open(AUDIT_PATH,  "a", encoding="utf-8") as f_audit:

    n_batches = (len(target_rows) + TASKS_PER_BATCH - 1) // TASKS_PER_BATCH
    for batch_idx in tqdm(range(n_batches)):
        batch_rows = target_rows[batch_idx * TASKS_PER_BATCH : (batch_idx + 1) * TASKS_PER_BATCH]
        batch_tasks = [
            {"task_id": r["task_id"], "prompt": r["text"], "tests": r["test_list"]}
            for r in batch_rows
        ]

        try:
            audits = process_tasks_batch(
                batch_tasks,
                n_samples=N_SAMPLES_PER_TASK,
                temperature=SAMPLE_TEMPERATURE,
                verbose=False,
            )
        except Exception as e:
            print(f"\n[batch starting at task {batch_tasks[0]['task_id']}] hard error: {e}")
            audits = [
                {
                    "task_id": t["task_id"], "status": "dropped",
                    "drop_reason": f"hard_error: {e}",
                    "samples": [], "n_samples": N_SAMPLES_PER_TASK,
                    "n_teacher_solved": 0, "n_student_solved": 0,
                    "n_saved": 0, "saved_examples": [],
                }
                for t in batch_tasks
            ]

        for audit in audits:
            f_audit.write(json.dumps(audit, ensure_ascii=False) + "\n")

            saved = audit.get("saved_examples", [])
            if saved:
                stats["tasks_saved"] += 1
            else:
                stats["tasks_dropped"] += 1
            for ex in saved:
                if ex["metadata"]["rescued_by_teacher"]:
                    stats["examples_teacher"] += 1
                else:
                    stats["examples_student"] += 1
                f_out.write(json.dumps(ex, ensure_ascii=False) + "\n")

            stats["total_teacher_solves"] += audit.get("n_teacher_solved", 0)
            stats["total_student_solves"] += audit.get("n_student_solved", 0)
            stats["total_samples_run"]    += audit.get("n_samples", 0)
            if any((s.get("teacher_drop_reason") == "teacher_no_code")
                   for s in audit.get("samples", [])):
                stats["teacher_no_code_retries"] += 1

        f_audit.flush()
        f_out.flush()

print("\nDone. Stats:", stats)
if stats["total_samples_run"]:
    yield_pct = 100.0 * stats["total_teacher_solves"] / stats["total_samples_run"]
    print(f"Teacher-correction yield per sample : {yield_pct:.1f}%")
    saved_total = stats["examples_teacher"] + stats["examples_student"]
    print(f"Examples saved : {saved_total} "
          f"({stats['examples_teacher']} teacher + {stats['examples_student']} student)")

## Audit: length distribution + max_seq_length sizing

Uses the Qwen3 tokenizer to compute training-token counts for every saved
example. The "would survive at Nk" table shows what fraction of the dataset
fits at each candidate `max_seq_length`.

In [ ]:
import numpy as np
from collections import Counter
from transformers import AutoTokenizer

audit_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID)

lengths = []
attempts_used = []
rescued = 0
student_only = 0

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        messages = [
            {"role": "user" if t["from"] == "human" else "assistant", "content": t["value"]}
            for t in ex["conversations"]
        ]
        text = audit_tokenizer.apply_chat_template(
            messages, tokenize=False, enable_thinking=True
        )
        n = len(audit_tokenizer.encode(text))
        lengths.append(n)
        attempts_used.append(ex["metadata"]["attempts_used"])
        if ex["metadata"]["rescued_by_teacher"]:
            rescued += 1
        else:
            student_only += 1

if not lengths:
    print("No examples in the output file yet. Run the bulk cell first.")
else:
    n_examples = len(lengths)
    arr = np.array(lengths)
    print(f"Examples saved : {n_examples}")
    print(f"  solved by student only : {student_only}")
    print(f"  rescued by teacher     : {rescued}")
    print()
    print("Token-length percentiles (training-accurate):")
    for p in (50, 75, 90, 95, 99):
        print(f"  p{p:>2d}  : {int(np.percentile(arr, p)):>6d}")
    print(f"  max  : {int(arr.max()):>6d}")
    print(f"  mean : {int(arr.mean()):>6d}")
    print()
    print("Would survive at max_seq_length:")
    for cap in (4096, 5120, 6144, 8192, 10240):
        survive = int((arr <= cap).sum())
        pct = 100.0 * survive / n_examples
        marker = "  <-- RECOMMENDED" if pct >= 95 and pct < 100 else ""
        print(f"  {cap:>5d} : {survive:>4d}/{n_examples} ({pct:5.1f}%){marker}")
    print()
    print(f"Attempts-used distribution: {dict(Counter(attempts_used))}")

    if os.path.exists(AUDIT_PATH):
        drop_reasons = Counter()
        n_audit = 0
        for line in open(AUDIT_PATH, "r", encoding="utf-8"):
            a = json.loads(line)
            n_audit += 1
            if a.get("status") == "dropped":
                drop_reasons[a.get("drop_reason", "unknown")] += 1
        print(f"\nProcessed tasks   : {n_audit}")
        print(f"Examples saved    : {n_examples}")
        print(f"Tasks dropped     : {sum(drop_reasons.values())}")
        print("Drop reasons:")
        for r, c in drop_reasons.most_common():
            print(f"  {c:>4d}  {r}")

    print("\n5 longest examples (candidates for inspection / filtering):")
    longest_idx = sorted(range(len(lengths)), key=lambda i: -lengths[i])[:5]
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        all_examples = [json.loads(line) for line in f]
    for i in longest_idx:
        ex = all_examples[i]
        print(f"  task_id={ex['task_id']:>4}  tokens={lengths[i]:>5}  attempts={ex['metadata']['attempts_used']}  rescued={ex['metadata']['rescued_by_teacher']}")